# RAG

Um modelo responde com o que está no prompt. RAG é a técnica de buscar, a cada pergunta, os trechos de uma base que provavelmente contêm a resposta, e colocá-los no prompt antes de gerar. Este notebook percorre as etapas: representar texto como vetor, indexar um documento, recuperar os trechos, gerar a resposta com a fonte e, por fim, deixar o agente decidir quando buscar.

In [ ]:
# No Google Colab, descomente e rode uma vez.

# !pip install -q langchain langchain-openai langchain-community langchain-chroma langgraph pdfplumber
# !pip install -q langchain-groq langchain-google-genai
# !pip install -q "agentkit @ git+https://github.com/silvaan/agentic-ai"

# import os
# from google.colab import userdata

# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
# os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
# os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
# os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

In [ ]:
import os
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image
from sklearn.decomposition import PCA

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.embeddings import init_embeddings
from langchain.messages import HumanMessage
from langchain.tools import tool
from langchain_chroma import Chroma
from langchain_community.document_loaders import PDFPlumberLoader
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import (
    CharacterTextSplitter,
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)

In [ ]:
model = init_chat_model("openai:gpt-4.1-mini", temperature=0.0)
embeddings = init_embeddings("openai:text-embedding-3-small")

# Outros modelos de embedding. Ao trocar, apague a pasta chroma_regimento, porque o tamanho do vetor muda.
# embeddings = init_embeddings("openai:openai/text-embedding-3-small", base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"], check_embedding_ctx_length=False)
# from agentkit import Embeddings
# embeddings = Embeddings("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

## Embeddings

Um embedding é um vetor que representa o significado de um texto. Textos próximos em significado ficam próximos no espaço, e é essa proximidade que a busca vai usar: em vez de procurar palavras iguais, ela procura vetores vizinhos.

In [ ]:
vector = embeddings.embed_query("A vacina reduziu os casos de gripe no inverno.")

print(len(vector))
print(vector[:5])

São 1536 números por texto, e o mesmo modelo de embedding sempre devolve vetores do mesmo tamanho. Os números em si não dizem nada; o que interessa é a distância entre dois vetores.

### Similaridade de cosseno

A medida usual é o cosseno do ângulo entre os dois vetores, que ignora o comprimento deles e olha só a direção:

$$\cos(a, b) = \frac{a \cdot b}{\|a\| \, \|b\|}$$

Vale 1 para vetores na mesma direção e 0 para vetores sem relação. A função abaixo recebe dois textos, calcula os dois embeddings e aplica a fórmula.

In [ ]:
def cosine_similarity(text_a: str, text_b: str) -> float:
    """Similaridade de cosseno entre os embeddings de dois textos."""
    a, b = embeddings.embed_query(text_a), embeddings.embed_query(text_b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


print(round(cosine_similarity("A vacina reduziu os casos de gripe no inverno.", "A imunização diminuiu as infecções respiratórias na estação fria."), 2))

Duas frases sem nenhuma palavra em comum além do artigo ficam a 0,76, porque dizem a mesma coisa. A tabela abaixo compara a mesma frase de referência com frases de vários assuntos.

In [ ]:
REFERENCE = "A vacina reduziu os casos de gripe no inverno."

SENTENCES = [
    "A imunização diminuiu as infecções respiratórias na estação fria.",
    "Médicos recomendam a vacinação anual contra a gripe.",
    "A inflação ficou abaixo da expectativa do mercado.",
    "A seleção venceu o amistoso por dois a zero.",
    "O filme venceu o prêmio de melhor direção no festival.",
    "Telescópio registra a galáxia mais distante já observada.",
]

pd.DataFrame(
    [(text, round(cosine_similarity(REFERENCE, text), 2)) for text in SENTENCES],
    columns=["frase", "similaridade"],
).sort_values("similaridade", ascending=False)

A paráfrase fica em 0,76 e a frase sobre vacinação contra a gripe, que traz outro fato do mesmo assunto, em 0,64. As frases de economia, esporte, cinema e ciência ficam todas abaixo de 0,35, e a ordem entre elas pouco importa: o embedding separa o que fala do mesmo assunto do que não fala.

### Vizinhança por assunto

Com mais textos, a proximidade organiza o espaço em regiões. As doze manchetes abaixo vêm de três assuntos, e a projeção dos 1536 números em duas dimensões, feita por PCA, deixa ver se o embedding as separa. A `QUERY` entra no mesmo gráfico como uma estrela.

In [ ]:
HEADLINES = {
    "ciência": [
        "Novo satélite brasileiro é lançado para monitorar queimadas",
        "Pesquisadores medem derretimento recorde no Ártico",
        "Telescópio registra galáxia mais distante já observada",
        "Vacina experimental contra dengue passa na fase três",
    ],
    "economia": [
        "Bolsa fecha em alta após anúncio de corte de juros",
        "Inflação de agosto fica abaixo da expectativa do mercado",
        "Governo lança linha de crédito para energia solar",
        "Dólar recua com entrada de capital estrangeiro",
    ],
    "esporte": [
        "Seleção brasileira vence amistoso por dois a zero",
        "Time local contrata técnico português para a próxima temporada",
        "Maratona de Natal bate recorde de inscritos",
        "Tenista brasileira avança às quartas em Paris",
    ],
}

In [ ]:
QUERY = "Cientistas descobrem planeta parecido com a Terra"

texts = [headline for items in HEADLINES.values() for headline in items] + [QUERY]
labels = [category for category, items in HEADLINES.items() for _ in items]
points = PCA(n_components=2).fit_transform(embeddings.embed_documents(texts))

COLORS = {"ciência": "#2a78d6", "economia": "#eb6834", "esporte": "#1baf7a"}
fig, ax = plt.subplots(figsize=(6, 4))
for category in HEADLINES:
    rows = [i for i, label in enumerate(labels) if label == category]
    ax.scatter(points[rows, 0], points[rows, 1], c=COLORS[category], s=60, label=category)
ax.scatter(*points[-1], c="black", marker="*", s=220, label="QUERY")
ax.legend(frameon=False, loc="lower center", bbox_to_anchor=(0.5, 1), ncol=4)
ax.set_axis_off()
plt.show()

Os três assuntos ocupam regiões diferentes, e a estrela caiu no meio das manchetes de ciência sem compartilhar nenhuma palavra com elas. Troque a `QUERY` por uma frase sobre juros ou sobre futebol e rode de novo: a estrela muda de região, e é essa distância que a busca vai calcular.

### Exercício 1

As dez sinopses abaixo estão em `MOVIES`. Escreva uma consulta sua, como um gênero ou uma situação, e ordene as sinopses pela `cosine_similarity` com ela. Qual filme ficou em primeiro, e a ordem faz sentido até que posição da lista?

In [ ]:
MOVIES = [
    "Um astronauta abandonado em Marte precisa cultivar comida para sobreviver até o resgate.",
    "Dois amigos de infância se reencontram em Lisboa e revivem um amor interrompido.",
    "Um detetive aposentado investiga o desaparecimento de uma pianista em Buenos Aires.",
    "Um robô de limpeza solitário encontra uma planta e muda o destino da humanidade.",
    "Soldados presos em uma praia esperam por barcos civis que cruzam o canal para salvá-los.",
    "Uma nadadora de cidade pequena treina para a seletiva olímpica contra a vontade da família.",
    "Uma família se muda para uma casa antiga e descobre que o porão guarda um segredo.",
    "Um professor de física tenta provar que viagens no tempo são possíveis usando um carro.",
    "Dois chefs rivais disputam a última estrela de um guia de restaurantes em Paris.",
    "Um garoto descobre que é um bruxo e vai estudar em uma escola escondida da vista de todos.",
]

# Seu código aqui

## Indexação

Indexar é preparar a base antes de qualquer pergunta, em três etapas: carregar o documento, quebrá-lo em trechos e guardar o embedding de cada trecho. O documento usado é o Regimento Geral da UFRN, um PDF público de mais de cem páginas.

### Loading

A unidade de texto no LangChain é o `Document`: um objeto com o texto em `page_content` e um dicionário livre em `metadata`. O metadado carrega o que não está no texto e vai ser preciso depois, como a origem, a página ou a data.

In [ ]:
manual = Document(
    page_content="O plenário do Departamento se reúne uma vez por mês.",
    metadata={"source": "anotação", "page": 1},
)

print(manual.page_content)
print(manual.metadata)

Um loader lê uma fonte e devolve uma lista de `Document` como esse, com os metadados que ele consegue extrair. Há um loader por tipo de fonte: `TextLoader` para arquivo de texto, `CSVLoader` para planilha, `WebBaseLoader` para página da web, `DirectoryLoader` para uma pasta inteira. Para PDF há mais de um, e o `PDFPlumberLoader` preserva bem o texto de páginas com colunas e tabelas.

In [ ]:
URL = "https://www.ufrn.br/resources/documentos/regimentos/RegimentoGeral.pdf"
FILE = Path("regimento.pdf")

if not FILE.exists():
    request = urllib.request.Request(URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=60) as response:
        FILE.write_bytes(response.read())

print(FILE.stat().st_size // 1024, "KB")

In [ ]:
docs = PDFPlumberLoader(str(FILE)).load()

print("documentos:", len(docs))
print("caracteres:", sum(len(doc.page_content) for doc in docs))
print("metadados:", {key: docs[30].metadata[key] for key in ("source", "page", "total_pages")})

O PDF virou uma lista de 139 `Document`, um por página, e o loader preencheu os metadados sozinho: o arquivo de origem, o número da página e o total. É esse metadado que permite dizer, no fim, de onde veio cada resposta.

Só que o loader conta as páginas a partir de 0, e o leitor de PDF e o número impresso no regimento contam a partir de 1. Como a página vai aparecer na resposta para ser conferida no PDF, a correção é feita aqui, uma vez, no metadado, e não em cada lugar que imprime a página.

In [ ]:
for doc in docs:
    doc.metadata["page"] += 1

print("página:", docs[67].metadata["page"])
print("primeira linha:", docs[67].page_content.splitlines()[0])

O documento de posição 67 na lista agora diz página 68, que é o número impresso na primeira linha do próprio texto e o que o leitor de PDF mostra. Daqui em diante, toda página citada pode ser aberta direto no PDF.

### Chunking

Uma página inteira é grande demais para recuperar: mistura assuntos e gasta contexto. Chunking é cortar o texto em trechos menores. Todo cortador recebe o `chunk_size`, tamanho máximo do trecho, e o `chunk_overlap`, quanto cada trecho repete do anterior. Os três cortadores abaixo são aplicados ao mesmo texto curto, em Markdown.

In [ ]:
TEXT = """# Biblioteca Central

## Horário

A biblioteca abre de segunda a sexta, das 7h às 22h, e aos sábados, das 8h às 12h. Nos feriados e no recesso entre semestres, o horário é reduzido.

## Empréstimo

Cada aluno pode levar até cinco livros por vez, por quinze dias na graduação e trinta na pós-graduação.

### Atrasos

O atraso na devolução gera suspensão: para cada dia de atraso, o aluno fica um dia sem poder pegar livros. A suspensão vale para todas as bibliotecas setoriais, e nenhum empréstimo pode ser renovado enquanto ela durar. Livros perdidos devem ser repostos com um exemplar da mesma edição ou de uma mais recente, e sem a reposição o aluno não consegue emitir o nada consta exigido na colação de grau.

## Salas de estudo

As salas em grupo comportam até oito pessoas e são reservadas pelo site."""

In [ ]:
def show_chunks(pieces: list[str]) -> None:
    """Imprime cada trecho inteiro, com o número e o tamanho."""
    for i, piece in enumerate(pieces):
        print(f"[{i}] {len(piece)} caracteres")
        print(piece)
        print()

O `CharacterTextSplitter` corta em um único separador, aqui a linha em branco, e junta as partes enquanto couberem em 300 caracteres. O parágrafo dos atrasos tem quase 400: tente prever o que acontece com ele.

In [ ]:
by_paragraph = CharacterTextSplitter(separator="\n\n", chunk_size=300, chunk_overlap=0)

show_chunks(by_paragraph.split_text(TEXT))

O parágrafo dos atrasos passou do limite, porque não tem linha em branco onde cortar.

O `RecursiveCharacterTextSplitter` tenta uma lista de separadores em ordem: parágrafo, linha, palavra e caractere. O que ainda passar do limite é cortado no separador seguinte. Aqui com 60 caracteres de sobreposição.

In [ ]:
recursive = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=60)

pieces = recursive.split_text(TEXT)
show_chunks(pieces)
print("fim do trecho 2   :", repr(pieces[2][-55:]))
print("começo do trecho 3:", repr(pieces[3][:55]))

O parágrafo dos atrasos foi cortado em palavra, e o trecho 3 repete o fim do 2: é a sobreposição, que deixa inteira, em um dos lados, a frase cortada na fronteira.

Os dois cortadores ignoram títulos: o "### Atrasos" ficou no trecho 1, longe do seu parágrafo. O `MarkdownHeaderTextSplitter` corta nos títulos.

In [ ]:
by_header = MarkdownHeaderTextSplitter([("#", "title"), ("##", "section"), ("###", "subsection")])

for piece in by_header.split_text(TEXT):
    print(piece.metadata)
    print(piece.page_content)
    print()

Cada seção virou um trecho, com os títulos nos metadados.

O regimento não tem essa estrutura. A página 68, como sai do loader:

In [ ]:
sample = docs[67].page_content

print(len(sample), "caracteres")
print(repr(sample))

Uma quebra a cada linha do PDF, nenhuma linha em branco, nenhum título. Resta o recursivo, que vai cortar em fim de linha. Para o índice, trechos de 1000 caracteres, um ou dois artigos, com 150 de sobreposição. O `split_documents` corta uma lista de `Document` e copia os metadados para cada trecho.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = text_splitter.split_documents(docs)

print("trechos:", len(chunks))
print("metadados preservados:", chunks[40].metadata["page"])

As 139 páginas viraram 166 trechos, cada um com a página de origem.

### Vector stores

Um vector store guarda cada trecho junto do seu embedding e responde à pergunta "quais vetores estão mais perto deste". O Chroma roda embutido no processo e grava em disco na pasta indicada. Há outros com a mesma interface: o `InMemoryVectorStore` do próprio LangChain, que vive só na memória e serve para experimentar, e o FAISS, uma biblioteca de busca vetorial rápida para índices grandes.

In [ ]:
vector_store = Chroma(
    collection_name="regimento",
    embedding_function=embeddings,
    persist_directory="./chroma_regimento",
    collection_metadata={"hnsw:space": "cosine"},
)

In [ ]:
ids = [f"chunk-{i}" for i in range(len(chunks))]
vector_store.add_documents(documents=chunks, ids=ids)

print("trechos indexados:", len(vector_store.get()["ids"]))

O `add_documents` calculou o embedding dos 166 trechos e gravou cada vetor junto do texto e dos metadados. Os `ids` fazem a operação ser idempotente: rodar a célula de novo substitui os mesmos 166 em vez de duplicá-los. O `hnsw:space` define a métrica da coleção como cosseno, a mesma da seção de embeddings; sem isso o Chroma usa distância euclidiana.

In [ ]:
reopened = Chroma(collection_name="regimento", embedding_function=embeddings, persist_directory="./chroma_regimento")

print("trechos no disco:", len(reopened.get()["ids"]))
print([doc.metadata["page"] for doc in reopened.similarity_search("trancamento de programa", k=3)])

Um objeto novo, criado só com o nome da coleção e a pasta, encontrou os 166 trechos gravados e respondeu à busca sem recalcular nenhum embedding. É o que separa um índice persistente de um em memória: o custo de indexar é pago uma vez, e a busca sobrevive ao reinício do processo.

### Exercício 2

Refaça o corte do regimento com `chunk_size` de 300 e de 2000, mantendo o mesmo `chunk_overlap`, e indexe cada versão em uma coleção própria do Chroma, com outro `collection_name`. Quantos trechos cada corte produziu, e em qual deles um artigo do regimento sobrevive inteiro dentro de um trecho?

In [ ]:
# Seu código aqui

## Retrieval

Recuperar é calcular o embedding da pergunta e devolver os `k` trechos mais próximos dela. O `k` é o orçamento: poucos trechos arriscam deixar a resposta de fora, muitos enchem o prompt de texto irrelevante e diluem o que importa.

A pergunta dos exemplos a seguir é de aluno: por quanto tempo dá para trancar o curso. O regimento responde no § 1º do art. 102, na página 68.

In [ ]:
for doc in vector_store.similarity_search("Por quantos períodos o aluno pode trancar o programa?", k=3):
    print(f'[página {doc.metadata["page"]}]')
    print(doc.page_content)
    print()

O primeiro trecho é o da página 68, e a página 69 vem logo depois com o resto do mesmo assunto. O `similarity_search` devolve uma lista de `Document`, do mais próximo ao mais distante, sem dizer o quanto cada um está perto. A versão com nota devolve pares de `Document` e número.

In [ ]:
for doc, score in vector_store.similarity_search_with_score("Por quantos períodos o aluno pode trancar o programa?", k=3):
    print(f'{score:.3f} | página {doc.metadata["page"]}')

A nota é a distância de cosseno entre a pergunta e o trecho, que vale 1 menos a similaridade da seção de embeddings: quanto menor, mais próximo. A célula abaixo faz uma pergunta que o regimento não responde.

In [ ]:
for doc, score in vector_store.similarity_search_with_score("Qual é o valor da bolsa de iniciação científica?", k=3):
    print(f'{score:.3f} | página {doc.metadata["page"]}')

Mesmo sem resposta no documento, voltaram três trechos, com as distâncias que houver. Aqui elas ficaram maiores, acima de 0,6, contra 0,27 do melhor trecho sobre trancamento, mas isso não vira um limiar: mais adiante, uma pergunta com resposta no regimento traz trechos errados a 0,36. A nota serve para ordenar os trechos de uma busca, e não para decidir se a busca deu certo.

### O que volta da busca

Cada item é o mesmo `Document` que saiu do loader e passou pelo cortador: o texto em `page_content`, o dicionário em `metadata` e o `id` que o índice recebeu.

In [ ]:
top = vector_store.similarity_search("Por quantos períodos o aluno pode trancar o programa?", k=1)[0]

print("campos:", sorted(vars(top)))
print("id:", top.id)
print("caracteres:", len(top.page_content))

O que o modelo vai receber é o trecho inteiro, do tamanho que o cortador definiu na indexação, e não a página nem a frase. O primeiro trecho impresso na busca acima começa no fim do art. 100 e termina no meio do § 4º do art. 102; a resposta, o § 1º, está no meio dele. Se o corte tivesse caído entre "concedido no" e "limite máximo de 04", o trecho chegaria ao prompt sem o número, o que liga a qualidade da resposta diretamente à decisão de chunking.

### Metadados

O `metadata` acompanha o trecho desde o loader. Parte vem do loader, parte das propriedades do PDF.

In [ ]:
docs[30].metadata

`source`, `page` e `total_pages` vêm do loader; o resto, do PDF. É um `dict` comum: dá para acrescentar chaves antes de indexar, e elas voltam com o trecho na busca.

Os metadados também filtram a busca. Pelo sumário, o regime didático, com matrícula, frequência e avaliação, ocupa as páginas 63 a 84. No regimento, "falta" também é infração disciplinar: tente prever de onde vêm os trechos sem o filtro.

In [ ]:
ABSENCES = "Quantas faltas posso ter numa matéria?"
didactic = {"$and": [{"page": {"$gte": 63}}, {"page": {"$lte": 84}}]}

print("sem filtro:", [doc.metadata["page"] for doc in vector_store.similarity_search(ABSENCES, k=3)])
print("com filtro:", [doc.metadata["page"] for doc in vector_store.similarity_search(ABSENCES, k=3, filter=didactic)])

Sem filtro, os trechos falam de falta como infração. Com o filtro, vem primeiro a página 74, onde o art. 115 reprova quem falta a 25% ou mais das atividades. O filtro só ajudou porque sabíamos onde procurar.

### O valor de k

A resposta da pergunta abaixo, o § 2º do art. 104, começa na página 69 e termina na 70, então ficou partida em dois trechos.

In [ ]:
for k in [1, 3, 6]:
    found = [doc.metadata["page"] for doc in vector_store.similarity_search("Quantas vezes posso trancar a mesma disciplina?", k=k)]
    print(f"k={k} -> páginas {found}")

Com `k=1` vem só a página 69, sem o número de vezes; a 70 entra com `k=3`. Com `k=6` entram trechos do regime disciplinar, que só ocupam o prompt.

### Exercício 3

Escreva uma pergunta sua sobre o regimento e recupere os trechos com `k` igual a 2 e a 6. Depois repita a busca com um filtro de metadado que restrinja a um intervalo de páginas. Quantos dos trechos recuperados têm mesmo a ver com a sua pergunta, e o filtro ajudou ou atrapalhou?

In [ ]:
# Seu código aqui

## Geração

A última etapa monta o prompt com os trechos recuperados e pede a resposta. O template deixa a estrutura visível: a instrução vai na mensagem de sistema, e o contexto e a pergunta entram como variáveis da mensagem humana.

In [ ]:
SYSTEM_TEMPLATE = """# Papel
Você responde perguntas sobre o Regimento Geral da UFRN a partir de trechos do documento.

# Regras
Responda usando apenas os trechos do contexto.
Termine a resposta com a página de onde ela saiu, entre parênteses, no formato (página 68).
Seja breve: no máximo três frases.
Se o contexto não bastar, diga que não encontrou e não cite página."""

HUMAN_TEMPLATE = """# Contexto
<contexto>
{context}
</contexto>

# Pergunta
<pergunta>
{question}
</pergunta>"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_TEMPLATE),
    ("human", HUMAN_TEMPLATE),
])


def format_docs(documents: list) -> str:
    """Junta os trechos em um texto só, com a página na frente de cada um."""
    return "\n\n".join(f"[página {doc.metadata['page']}] {doc.page_content}" for doc in documents)

In [ ]:
question = "Por quantos períodos o aluno pode trancar o programa?"
documents = vector_store.similarity_search(question, k=3)

filled = prompt.invoke({"context": format_docs(documents), "question": question})

for message in filled.messages:
    print(f"[{message.type}]")
    print(message.content)
    print()

É isso que o modelo recebe: o papel e as regras na mensagem de sistema, e os três trechos e a pergunta na mensagem humana, cada um em sua seção e entre tags, que separam o texto do documento das instruções. A página só pode ser citada porque está escrita na frente de cada trecho; o modelo não vê os metadados, só o texto que o `format_docs` monta. Quatro instruções da mensagem de sistema mudam o resultado: responder só com o contexto, terminar com a página, ser breve e dizer quando o contexto não basta, sem página nesse caso.

In [ ]:
chain = prompt | model | StrOutputParser()


def answer(question: str, k: int = 3) -> str:
    """Recupera os trechos e gera a resposta citando a página."""
    documents = vector_store.similarity_search(question, k=k)
    return chain.invoke({"context": format_docs(documents), "question": question})

In [ ]:
print(answer("Por quantos períodos o aluno pode trancar o programa?"))

A resposta veio do documento e termina com a página 68: abra o PDF nela e confira o § 1º do art. 102. Esse é o ganho do RAG sobre perguntar direto ao modelo: a afirmação fica rastreável até a fonte, e a página impressa no regimento é a mesma que o leitor de PDF mostra, por causa da correção feita no loading.

In [ ]:
print(answer("Quantas vezes posso trancar a mesma disciplina?"))

Com `k=3`, o trecho da página 70 chegou ao prompt, e a resposta cita a 70, onde o número de vezes está, e não a 69, que foi o trecho mais próximo. A instrução de citar a página faz o modelo apontar o trecho que usou, e não só o primeiro da lista.

In [ ]:
print(answer("Qual é o valor da bolsa de iniciação científica?"))

Aqui a busca trouxe três trechos e nenhum fala do valor de bolsa, então o modelo diz que não encontrou em vez de inventar, e não cita página. As duas partes da instrução importam: sem a de admitir a ausência, ele preencheria a lacuna com um valor plausível e errado; sem a de não citar página nesse caso, a regra de sempre terminar com a página o leva a anexar a de um trecho qualquer, o que dá uma falsa aparência de fonte.

### Exercício 4

Use o `answer` com duas perguntas suas: uma cuja resposta você sabe que está no regimento e outra que sabe que não está. Confira se a página citada na primeira contém mesmo a resposta, e o que o modelo fez na segunda.

In [ ]:
# Seu código aqui

## O retriever como ferramenta

Na função `answer` a busca acontece sempre, mesmo quando a pergunta não precisa dela. Embrulhando o retriever em uma ferramenta e entregando a um agente, a busca vira uma decisão do modelo: ele chama quando julga que precisa do documento e responde direto quando não precisa. O `create_agent` monta o laço de ferramentas; o que é deste notebook é a ferramenta.

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})


@tool
def search_regiment(query: str) -> str:
    """Busca trechos do Regimento Geral da UFRN relacionados à consulta."""
    return format_docs(retriever.invoke(query))

In [ ]:
SYSTEM = """# Papel
Você responde dúvidas de alunos sobre a UFRN.

# Regras
Para perguntas sobre o regimento, use a ferramenta e termine a resposta com a página, no formato (página 68).
Para o resto, responda direto.
Seja breve: no máximo três frases."""

rag_agent = create_agent(model, tools=[search_regiment], system_prompt=SYSTEM)
Image(rag_agent.get_graph().draw_mermaid_png())

In [ ]:
result = rag_agent.invoke({"messages": [HumanMessage("Por quantos períodos o aluno pode trancar o programa?")]})

for message in result["messages"]:
    print(f"[{message.type}]")
    print(message.content or message.tool_calls)
    print()

A conversa tem quatro mensagens: a pergunta, a decisão do modelo de chamar `search_regiment` com uma consulta escrita por ele, a `ToolMessage` com os três trechos que a ferramenta devolveu, com as páginas, e a resposta final, que cita a página tirada desses trechos. O grafo é o mesmo agente de duas caixas, `model` e `tools`; a diferença está no que a ferramenta faz quando é chamada, que agora é consultar o índice.

### Exercício 5

Rode o `rag_agent` com uma pergunta cuja resposta não está no regimento e observe a lista de mensagens: ele buscou? O que respondeu? Depois mude o `SYSTEM` para pedir que ele tente uma segunda busca com outras palavras antes de desistir, e diga o que mudou na lista.

In [ ]:
# Seu código aqui